# 09. 최종데이터 병합

메인 49개 지역에 어린이사고·어린이인구·어린이보호구역 데이터를 결합합니다.
과속방지턱은 데이터가 공통으로 존재하는 35개 지역에 대해 별도 데이터셋을 만듭니다.

**출력**
- `../data_processed/final_main_49.csv`
- `../data_processed/final_speed_bump_35.csv`

In [ ]:
import pandas as pd

main_49 = pd.read_csv(
    "../data_processed/main_analysis_49_regions.csv"
)
speed_35 = pd.read_csv(
    "../data_processed/speed_bump_analysis_35_regions.csv"
)

accident = pd.read_csv(
    "../data_clean/accident_2020_2025_child_clean.csv"
)
population = pd.read_csv(
    "../data_clean/population_2025_child_clean.csv"
)
schoolzone = pd.read_csv(
    "../data_clean/schoolzone_2026_clean.csv"
)

print("메인 지역:", len(main_49))
print("과속방지턱 지역:", len(speed_35))

## 1) 병합용 지역키 통일

In [ ]:
def normalize_region_name(x):
    if pd.isna(x):
        return x

    x = str(x).strip()
    x = x.replace("강원도 ", "강원특별자치도 ")
    x = x.replace("전라북도 ", "전북특별자치도 ")

    if x == "세종특별자치시":
        x = "세종특별자치시 세종시"

    return x

main_49["sido_sgg_nm"] = (
    main_49["SIDO_NM"].astype(str).str.strip()
    + " "
    + main_49["SIGUNGU_NM"].astype(str).str.strip()
)

for df in [main_49, accident, population, schoolzone]:
    df["sido_sgg_nm"] = (
        df["sido_sgg_nm"]
        .apply(normalize_region_name)
    )

## 2) 2020~2025 어린이사고 총합

In [ ]:
accident_total = (
    accident
    .groupby("sido_sgg_nm", as_index=False)
    .agg(
        child_accident_count=("acc_cnt", "sum"),
        child_death_count=("dth_dnv_cnt", "sum"),
        child_injury_count=("injpsn_cnt", "sum")
    )
)

display(accident_total.head())

## 3) 메인 49개 지역 최종 병합

In [ ]:
final_main = (
    main_49
    .merge(
        accident_total,
        on="sido_sgg_nm",
        how="left"
    )
    .merge(
        population,
        on="sido_sgg_nm",
        how="left"
    )
    .merge(
        schoolzone,
        on="sido_sgg_nm",
        how="left"
    )
)

print("지역 수:", len(final_main))
print("\n병합 후 결측치")
print(final_main.isna().sum())

## 4) CCTV 변수 정리

`cctv_count_total`은 일부 지역에서 설치대수 자체가 미제공이라 메인 분석에서 제외합니다.
대신 결측이 없는 `CCTV 설치 보호구역 수 / 전체 보호구역 수`를 사용합니다.

In [ ]:
final_main = final_main.drop(
    columns=["cctv_count_total"]
)

final_main["cctv_install_rate"] = (
    final_main["cctv_installed_zone_count"]
    / final_main["schoolzone_count"]
)

final_main["cctv_install_rate"] = (
    final_main["cctv_install_rate"]
    .fillna(0)
)

print("최종 결측치")
print(final_main.isna().sum())

assert len(final_main) == 49
assert final_main.isna().sum().sum() == 0

In [ ]:
final_main.to_csv(
    "../data_processed/final_main_49.csv",
    index=False,
    encoding="utf-8-sig"
)

print("메인 데이터 저장 완료:", len(final_main))
display(final_main.head())

## 5) 과속방지턱 추가분석 35개 지역

In [ ]:
final_speed_35 = final_main.merge(
    speed_35[
        ["SIDO_NM", "SIGUNGU_NM", "speed_bump_count"]
    ],
    on=["SIDO_NM", "SIGUNGU_NM"],
    how="inner"
)

print("과속방지턱 추가분석 지역:", len(final_speed_35))
print(final_speed_35.isna().sum())

assert len(final_speed_35) == 35
assert final_speed_35.isna().sum().sum() == 0

In [ ]:
final_speed_35.to_csv(
    "../data_processed/final_speed_bump_35.csv",
    index=False,
    encoding="utf-8-sig"
)

print("추가분석 데이터 저장 완료")